# Latent-Space Noise Injection: Mechanism, Effect, and Theory

Visualizes the latent-space noise injection used during DGD training (`training.latent_noise_enabled`/`_start`/`_end`), reading a completed run's saved checkpoints -- **no training happens here**.

**Mechanism:** before the decoder sees $z$, isotropic noise is added: $\tilde z = z + \epsilon,\ \epsilon \sim \mathcal{N}(0, \sigma_t^2 I)$, with $\sigma_t$ annealed from a large value down to a small one (`cosine_noise_schedule`). The decoder only ever trains against $\tilde z$, never clean $z$.

**Why:** the DGD has no encoder or reparameterization trick -- $z$ is optimized directly via MAP, so nothing forces the decoder to behave sensibly off the exact training points. Training against a noisy neighborhood of each $z_i$ pushes the decoder toward a smooth mapping (the same principle as denoising autoencoders), while annealing $\sigma_t \to 0$ lets representations settle into precise, non-overlapping positions the GMM prior can separate.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import torch
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from hydra import initialize, compose

current_dir = Path.cwd()
if 'notebooks' in current_dir.parts:
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.append(str(project_root))
sys.path.append(str(project_root / 'src'))

from src.models import ConvDecoder
from src.utils import setup_device, set_random_seed, setup_cuml_acceleration
from src.utils.checkpoint import load_checkpoint
from src.utils.schedules import cosine_noise_schedule
from src.visualization import plot_noise_comparison

device = setup_device(verbose=True)
set_random_seed(seed=42, device=device)
setup_cuml_acceleration(verbose=True)


In [ ]:
with initialize(version_base=None, config_path="../config"):
    config = compose(config_name="config")

config.paths.experiments_dir = str(project_root / "experiments")

experiments_dir = Path(config.paths.experiments_dir)
candidates = sorted(
    p for p in experiments_dir.glob(f"*_{config.experiment_name}")
    if (p / "models" / "best").is_dir()
)
assert candidates, (
    f"No completed training runs found under {experiments_dir} matching "
    f"*_{config.experiment_name} (looked for a models/best/ subfolder). "
    "Run dgd_training_demo.ipynb first."
)
run_dir = candidates[-1]  # newest, since the timestamp prefix sorts lexicographically
print(f"Using training run: {run_dir}")

# This run's own saved config, not the live config/config.yaml -- so the
# decoder architecture and noise schedule below always match what this run
# was actually trained with, even if config.yaml has changed since.
trained_cfg = OmegaConf.load(run_dir / "config.yaml")


In [ ]:
def decoder_factory():
    return ConvDecoder(
        latent_dim=trained_cfg.model.representation.n_features,
        hidden_dims=trained_cfg.model.decoder.hidden_dims,
        output_channels=trained_cfg.model.decoder.output_channels,
        output_size=trained_cfg.model.decoder.output_size,
        activation=trained_cfg.model.decoder.activation,
        final_activation=trained_cfg.model.decoder.final_activation,
        dropout_rate=trained_cfg.model.decoder.dropout_rate,
        init_size=trained_cfg.model.decoder.init_size,
    )

checkpoint_dirs = sorted(
    (run_dir / "models" / "checkpoints").glob("epoch_*"),
    key=lambda p: int(p.name.split('_')[1]),
)
checkpoint_epochs = [int(p.name.split('_')[1]) for p in checkpoint_dirs]
print(f"Found {len(checkpoint_dirs)} saved checkpoints at epochs: {checkpoint_epochs}")

noise_start = trained_cfg.training.get('latent_noise_start', 1.0)
noise_end = trained_cfg.training.get('latent_noise_end', 0.01)
noise_enabled = trained_cfg.training.get('latent_noise_enabled', False)
total_epochs = trained_cfg.training.epochs
print(f"Noise schedule for this run: enabled={noise_enabled}, start={noise_start}, "
      f"end={noise_end}, over {total_epochs} epochs")


## The noise schedule

$$
\sigma_t = \sigma_{\text{end}} + (\sigma_{\text{start}} - \sigma_{\text{end}}) \cdot \frac{1 + \cos(\pi p_t)}{2}, \qquad p_t = \mathrm{clip}\!\left(\frac{t-1}{T-1},\, 0,\, 1\right)
$$

Cosine anneal from `noise_start` at epoch 1 to `noise_end` at the final epoch -- fast to drop at first, flattening out later. Dots mark the checkpoints visualized below.

In [ ]:
epochs_range = list(range(0, total_epochs + 1))
schedule = [cosine_noise_schedule(e, total_epochs, noise_start, noise_end) if noise_enabled else 0.0
            for e in epochs_range]
checkpoint_sigmas = [cosine_noise_schedule(e, total_epochs, noise_start, noise_end) if noise_enabled else 0.0
                     for e in checkpoint_epochs]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_range, schedule, color='orange', linewidth=2, label='noise scale (sigma)')
ax.scatter(checkpoint_epochs, checkpoint_sigmas, color='black', zorder=5, label='saved checkpoints')
for e, s in zip(checkpoint_epochs, checkpoint_sigmas):
    ax.annotate(f"epoch {e}\nsigma={s:.4f}", (e, s), textcoords="offset points",
                xytext=(0, 10), fontsize=8, ha='center')
ax.set_xlabel('Epoch')
ax.set_ylabel('Noise scale (sigma)')
ax.set_title(f'Noise Schedule -- {run_dir.name}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## What the noise actually does to the representations

For each checkpoint, `plot_noise_comparison` shows two views in a fresh PCA projection of that checkpoint's train representations (PCA, not UMAP/t-SNE, since only a linear projection makes displacement in the picture faithful to displacement in the real space):

- **Left** -- clean $z$ (blue) vs. $\tilde z$ with that checkpoint's $\sigma_t$ (red). Red spilling into the gaps between clusters means noise is filling that space.
- **Right** -- one fixed point (`point_index=0`) with 200 noise draws, annotated with $\sigma_t$, $\sigma_t\sqrt{d}$ (true expected displacement in $d$-dim space -- a 2D PCA projection alone would understate it), the median nearest-neighbor distance in raw representation space, and their **ratio**. The ratio is what's comparable *across* checkpoints, since PCA is refit independently each time and its axes can rotate between epochs.

**What to look for:** ratio well above 1 early (noise spans neighboring clusters), dropping well below 1 by the end (noise ball stays inside one cluster's own density).

(Epoch 0 starts every representation at the same point under `distribution: "zeros"`, so PCA variance is `nan%` and NN distance is `0` -- expected for a zero-variance start, not a bug.)

In [ ]:
for epoch, ckpt_dir in zip(checkpoint_epochs, checkpoint_dirs):
    checkpoint = load_checkpoint(ckpt_dir, decoder_factory, device=device)
    rep = checkpoint['rep']
    sigma = cosine_noise_schedule(epoch, total_epochs, noise_start, noise_end) if noise_enabled else 0.0

    plot_noise_comparison(
        representations=rep.z.detach(),
        noise_scale=sigma,
        point_index=0,
        title=f"Train Representations -- epoch {epoch} (sigma={sigma:.4f})",
        show=True,
    )


## Practical takeaway for tuning `noise_start` / `noise_end`

Change `training.latent_noise_*` in `config/config.yaml`, run a fresh training run, then re-run this notebook and watch the `ratio` across checkpoints:

- Ratio never exceeds ~1 anywhere → noise probably isn't spreading representations far enough → raise `latent_noise_start`.
- Ratio still well above 1 at the final checkpoint → representations haven't settled → lower `latent_noise_end`, extend training, or slow the anneal.

The same figures are also generated automatically every 50 epochs during normal training and every 50 steps during `dgd_test_inference.ipynb`'s Algorithm-2 optimization (`figures/training/noise_*.png`). This notebook is just a narrated walkthrough of the same plots.